In [1]:
%pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [1]:
%pip install pennylane pennylane-lightning jupyter jax jaxlib optax scikit-learn scikit-image pybind11

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pennylane as qml
import jax
import jax.numpy as jnp
import optax
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

# ----------------------------------------------
# [데이터셋 전처리부] 
# ----------------------------------------------
def prepare_quantum_dataset(classes=(3,6), test_size=0.2, random_state=42):
    digits = load_digits()
    X_raw, Y_raw = digits.data, digits.target
    
    mask = np.isin(Y_raw, classes)
    X_filtered, Y_filtered = X_raw[mask], Y_raw[mask]
    Y_binary = np.where(Y_filtered == classes[0], 1.0, -1.0)

    n_samples = X_filtered.shape[0]
    X_resized = np.zeros((n_samples, 16))

    for i in range(n_samples):
        img_8x8 = X_filtered[i].reshape(8, 8)
        img_4x4 = img_8x8.reshape(4, 2, 4, 2).mean(axis=(1, 3))
        img_binary = np.where(img_4x4 > img_4x4.mean(), 1.0, 0.0)
        X_resized[i] = img_binary.flatten()

    X_train, X_test, Y_train, Y_test = train_test_split(
        X_resized, Y_binary, test_size=test_size, random_state=random_state, stratify=Y_binary
    )
    return X_train, X_test, Y_train, Y_test

# ----------------------------------------------
# [퀀텀 컴퓨팅 부] JAX 인터페이스 적용
# ----------------------------------------------
n_qubits = 17
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="jax", diff_method="backprop")
def quantum_neural_net(inputs, weights):
    # 1. 16개의 픽셀 데이터 인코딩
    for i in range(len(inputs)):
        qml.RX(inputs[i] * jnp.pi, wires=i)
    
    # 2. 17개의 파라미터(Weights) 학습
    for i in range(n_qubits):
        qml.RY(weights[i], wires=i)

    # 3. 얽힘 (0->1->2...->16 방향으로 정보가 흐름)
    for i in range(n_qubits-1):
        qml.CNOT(wires=[i, i+1])
    
    # 모든 정보가 모이는 마지막 큐비트(n_qubits-1)를 측정해야 함
    # (기존 코드는 PauliZ(0)을 측정하고 있었는데, CNOT 체인에서 wire 0은
    #  control로만 쓰이고 target이 된 적이 없어 reduced state가 변하지 않음.
    #  즉 weights[1:]와 나머지 15개 픽셀에 대한 gradient가 전부 0이 되어
    #  학습이 사실상 진행되지 않았음.)
    return qml.expval(qml.PauliZ(n_qubits - 1))

# 배치 처리를 위한 vmap
batched_qnn = jax.vmap(quantum_neural_net, in_axes=(0, None))

# ----------------------------------------------
# [고전 컴퓨팅 부] 손실 함수 및 JIT 컴파일 루프
# ----------------------------------------------
def loss_function(weights, X, Y):
    predictions = batched_qnn(X, weights)
    return jnp.mean((predictions - Y) ** 2)

# 🚨 수정됨: 미분 함수 선언
loss_and_grad_fn = jax.value_and_grad(loss_function)

# 🚨 수정됨: JAX 전용 초고속 업데이트 함수 (JIT 컴파일 적용)
# 파이썬 for문 안에서 이걸 호출하면 메모리 누수 없이 엄청나게 빠르게 연산됩니다.
@jax.jit
def update_step(opt_state, theta, X_batch, Y_batch):
    loss_val, grads = loss_and_grad_fn(theta, X_batch, Y_batch)
    updates, new_opt_state = optimizer.update(grads, opt_state)
    new_theta = optax.apply_updates(theta, updates)
    return new_theta, new_opt_state, loss_val

# ---------------------------------------------------------
# [실제 수행] 최적화 루프
# ---------------------------------------------------------
X_train, X_test, Y_train, Y_test = prepare_quantum_dataset()

# 데이터셋을 JAX 배열로 변환
X_train_jax = jnp.array(X_train)
Y_train_jax = jnp.array(Y_train)
X_test_jax = jnp.array(X_test)

# 초기 파라미터 세팅
key = jax.random.PRNGKey(42)
theta = jax.random.normal(key, (n_qubits,))

# 최적화기 세팅
learning_rate = 0.1
optimizer = optax.sgd(learning_rate)
opt_state = optimizer.init(theta)

epochs = 100
# batch_size = 32 # 🚨 수정됨: 메모리 폭발을 막는 안전한 배치 사이즈
# n_batches = len(X_train_jax) // batch_size

# print(f"\n🚀 JAX Mini-Batch 학습 시작 (데이터 {len(X_train)}개, 배치 {batch_size}개씩)...")

# for epoch in range(epochs):
#     epoch_loss = 0.0
    
#     # 🚨 미니 배치 루프 (11GB 메모리 폭발 방지)
#     for b in range(n_batches):
#         start_idx = b * batch_size
#         end_idx = start_idx + batch_size
#         X_batch = X_train_jax[start_idx:end_idx]
#         Y_batch = Y_train_jax[start_idx:end_idx]
        
#         # JIT 적용된 업데이트 함수 호출
#         theta, opt_state, loss_val = update_step(opt_state, theta, X_batch, Y_batch)
#         epoch_loss += loss_val
        
#     avg_loss = epoch_loss / n_batches
    
#     if (epoch+1) % 10 == 0:
#         print(f"Epoch: {epoch+1:3d} | Avg Loss: {avg_loss:.4f}")

print("\n🚀 Full-Batch JAX 학습 시작...")
for epoch in range(epochs):
    # 단 한 줄로 전체 데이터에 대한 손실값과 미분값을 구함! (초고속)
    current_loss, grads = loss_and_grad_fn(theta, X_train_jax, Y_train_jax)

    # 파라미터 업데이트
    updates, opt_state = optimizer.update(grads, opt_state)
    theta = optax.apply_updates(theta, updates)

    if (epoch+1) % 10 == 0:
        print(f"Epoch: {epoch+1:3d} | Loss: {current_loss:.4f}")

print("\n학습 완료!")

# 테스트 검증
Y_test_pred = batched_qnn(X_test_jax, theta)
predicted_labels = jnp.where(Y_test_pred > 0, 1, -1)
accuracy = jnp.mean(predicted_labels == jnp.array(Y_test))
print(f"테스트 정확도: {accuracy * 100:.2f}%")

## 실제 IBM Qiskit 인스턴스(QPU)로 학습하는 버전

위 셀은 PennyLane + JAX(`default.qubit`, backprop, jit)로 시뮬레이터에서만 도는 버전입니다.
아래는 **학습 루프 전체를 Qiskit으로 전환**해서 실제 IBM 무료(Open) 플랜 QPU(`least_busy` 백엔드)로 돌리는 버전입니다.

무료 플랜은 **월 10분의 QPU 실행 시간**만 제공되기 때문에, 위 버전과 동일하게 갈 수 없어 다음과 같이 규모를 크게 줄였습니다.

- **큐비트 수**: 17 → 5 (입력 4픽셀 + 결과 취합용 큐비트 1개, 원본과 동일한 비율/구조)
- **입력 특징**: 8x8 이미지를 4x4가 아니라 **2x2(4픽셀)** 로 풀링
- **데이터 수**: 학습 12개 / 테스트 6개 (3 vs 6 클래스에서 소량만 샘플링)
- **에폭 수**: 5
- **그래디언트 방식**: `backprop`/parameter-shift 대신 **SPSA**(Simultaneous Perturbation Stochastic Approximation) 사용
  - parameter-shift는 파라미터 1개당 회로 2개가 필요해서(5개 파라미터 → 스텝당 10개 이상) 실기기에서 비용이 큼
  - SPSA는 파라미터 개수와 무관하게 **스텝당 회로 2세트(plus/minus)** 만 있으면 됨
  - 게다가 배치 안의 모든 샘플을 **한 Job의 한 PUB**(parameter_values를 2차원 배열로)에 담아 보내므로, **에폭당 IBM Runtime Job이 정확히 1개**만 생성됩니다 → 학습 5 에폭 + 최종 테스트 평가 1회 = 총 6개 Job

`QISKIT_APIKEY` 환경변수가 설정되어 있어야 하며(`qiskit_setup.ipynb`와 동일한 방식), `channel="ibm_quantum_platform"`으로 연결합니다.

⚠️ 아래 상수(`EPOCHS`, `N_TRAIN`, `N_TEST`, `SHOTS`)를 늘리면 10분 무료 할당량을 초과할 수 있으니, 먼저 기본값 그대로 한 번 실행해보고 [IBM Quantum 대시보드](https://quantum.cloud.ibm.com/)에서 사용량을 확인한 뒤 조절하세요.


In [ ]:
%pip install qiskit qiskit-ibm-runtime

In [ ]:
import os
import time

from qiskit_ibm_runtime import QiskitRuntimeService

QISKIT_APIKEY = os.getenv("QISKIT_APIKEY")
if not QISKIT_APIKEY:
    raise RuntimeError("QISKIT_APIKEY 환경변수가 설정되어 있지 않습니다. (qiskit_setup.ipynb 참고)")

service = QiskitRuntimeService(
    token=QISKIT_APIKEY,
    channel="ibm_quantum_platform",
)

# 무료(Open) 플랜에서 사용 가능한, 현재 대기열이 가장 짧은 실제 QPU를 선택
backend = service.least_busy(operational=True, simulator=False)
print(f"선택된 백엔드: {backend.name} (큐비트 수: {backend.num_qubits})")

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

# ----------------------------------------------
# [데이터셋 전처리부] 무료 QPU 예산에 맞춰 2x2(4픽셀)로 축소
# ----------------------------------------------
N_FEATURES = 4     # 2x2 풀링 결과
N_QUBITS = N_FEATURES + 1  # 입력 4개 + 결과 취합용 큐비트 1개
N_TRAIN = 12        # 클래스당 6개
N_TEST = 6          # 클래스당 3개
EPOCHS = 5
SHOTS = 256

def prepare_quantum_dataset_small(classes=(3, 6), n_train=N_TRAIN, n_test=N_TEST, random_state=42):
    digits = load_digits()
    X_raw, Y_raw = digits.data, digits.target

    mask = np.isin(Y_raw, classes)
    X_filtered, Y_filtered = X_raw[mask], Y_raw[mask]
    Y_binary = np.where(Y_filtered == classes[0], 1.0, -1.0)

    n_samples = X_filtered.shape[0]
    X_resized = np.zeros((n_samples, N_FEATURES))
    for i in range(n_samples):
        img_8x8 = X_filtered[i].reshape(8, 8)
        img_2x2 = img_8x8.reshape(2, 4, 2, 4).mean(axis=(1, 3))
        img_binary = np.where(img_2x2 > img_2x2.mean(), 1.0, 0.0)
        X_resized[i] = img_binary.flatten()

    n_needed = n_train + n_test
    X_small, _, Y_small, _ = train_test_split(
        X_resized, Y_binary, train_size=n_needed, random_state=random_state, stratify=Y_binary
    )
    X_train, X_test, Y_train, Y_test = train_test_split(
        X_small, Y_small, train_size=n_train, random_state=random_state, stratify=Y_small
    )
    return X_train, X_test, Y_train, Y_test

X_train, X_test, Y_train, Y_test = prepare_quantum_dataset_small()
print(f"학습 샘플: {len(X_train)}개, 테스트 샘플: {len(X_test)}개, 특징 수: {N_FEATURES}")

In [ ]:
# ----------------------------------------------
# [퀀텀 회로부] native Qiskit으로 재구성한 동일한 구조의 QNN
# (입력 RX 인코딩 -> 가중치 RY -> CNOT 체인 -> 마지막 큐비트 PauliZ 측정)
# ----------------------------------------------
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

x_params = ParameterVector("x", N_FEATURES)
theta_params = ParameterVector("theta", N_QUBITS)

qc = QuantumCircuit(N_QUBITS)
for i in range(N_FEATURES):
    qc.rx(x_params[i] * np.pi, i)
for i in range(N_QUBITS):
    qc.ry(theta_params[i], i)
for i in range(N_QUBITS - 1):
    qc.cx(i, i + 1)

# 마지막 큐비트(N_QUBITS-1)를 측정: CNOT 체인에서 실제로 모든 정보가 모이는 큐비트
pauli_str = "Z" + "I" * (N_QUBITS - 1)
observable = SparsePauliOp.from_list([(pauli_str, 1.0)])

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)
isa_observable = observable.apply_layout(isa_circuit.layout)

# transpile 후 파라미터 이름 -> 회로 상 위치를 미리 계산해둠 (parameter_values 배열 순서를 맞추기 위함)
param_names = [p.name for p in isa_circuit.parameters]
print(f"회로 파라미터 순서: {param_names}")
print(f"변환된(ISA) 회로 깊이: {isa_circuit.depth()}")

In [ ]:
# ----------------------------------------------
# [고전 컴퓨팅 부] SPSA로 실제 QPU 위에서 학습
# ----------------------------------------------
import re
from qiskit_ibm_runtime import EstimatorV2 as Estimator

estimator = Estimator(backend)
estimator.options.default_shots = SHOTS

def _param_row(x_sample, theta_vec):
    """isa_circuit.parameters 순서(param_names)에 맞춰 하나의 파라미터 값 행을 만든다."""
    row = []
    for name in param_names:
        vec_name, idx = re.match(r"(\w+)\[(\d+)\]", name).groups()
        idx = int(idx)
        row.append(x_sample[idx] if vec_name == "x" else theta_vec[idx])
    return row

def batched_expectations(theta_vec, X_batch):
    """X_batch의 모든 샘플을 하나의 Job(하나의 PUB)으로 묶어서 실제 QPU에 보낸다."""
    values = [_param_row(x, theta_vec) for x in X_batch]
    pub = (isa_circuit, isa_observable, np.array(values))
    job = estimator.run([pub])
    result = job.result()[0]
    return np.array(result.data.evs)

def loss_from_preds(preds, Y_batch):
    return float(np.mean((preds - Y_batch) ** 2))

# SPSA 하이퍼파라미터 (표준 decaying gain sequence)
rng = np.random.default_rng(42)
theta = rng.normal(scale=0.5, size=N_QUBITS)

a0, c0 = 0.3, 0.2
A = 0.1 * EPOCHS + 1
alpha, gamma = 0.602, 0.101

print("\n🚀 실제 QPU(SPSA) 학습 시작...")
job_count = 0
for k in range(1, EPOCHS + 1):
    a_k = a0 / (k + A) ** alpha
    c_k = c0 / k ** gamma
    delta = rng.choice([-1.0, 1.0], size=N_QUBITS)

    theta_plus = theta + c_k * delta
    theta_minus = theta - c_k * delta

    t0 = time.time()
    # plus/minus 파라미터 세트를 전부 합쳐 한 Job(한 PUB)으로 제출
    combined_theta = np.stack([theta_plus, theta_minus])  # SPSA 두 세트를 순서대로 배치와 묶어서 보냄
    values = []
    for tv in combined_theta:
        values.extend(_param_row(x, tv) for x in X_train)
    pub = (isa_circuit, isa_observable, np.array(values))
    job = estimator.run([pub])
    evs = np.array(job.result()[0].data.evs)
    job_count += 1
    elapsed = time.time() - t0

    preds_plus, preds_minus = evs[: len(X_train)], evs[len(X_train):]
    loss_plus = loss_from_preds(preds_plus, Y_train)
    loss_minus = loss_from_preds(preds_minus, Y_train)

    ghat = (loss_plus - loss_minus) / (2 * c_k) * delta
    theta = theta - a_k * ghat

    avg_loss = (loss_plus + loss_minus) / 2
    print(f"Epoch {k:2d} | avg Loss: {avg_loss:.4f} | job #{job_count} 소요시간: {elapsed:.1f}s")

print("\n학습 완료! 최종 theta:", theta)

In [ ]:
# ----------------------------------------------
# [실제 QPU 최종 검증] 테스트셋 전체를 한 Job으로 평가
# ----------------------------------------------
Y_test_pred = batched_expectations(theta, X_test)
job_count += 1
predicted_labels = np.where(Y_test_pred > 0, 1.0, -1.0)
accuracy = np.mean(predicted_labels == Y_test)
print(f"실제 QPU({backend.name}) 테스트 정확도: {accuracy * 100:.2f}%  (총 Job 수: {job_count})")